# ¿Se paga por la edad o por el rendimiento?
## Mercado de futbolistas en las cinco grandes ligas

Memoria del análisis · Proyecto individual de EDA · Data Science, The Bridge  
Autor: Patricio · Entrega: 6 de noviembre de 2026

## 1. Pregunta y objetivo

El valor de mercado de un futbolista puede seguir a dos cosas distintas: su **edad** (el potencial, los años de carrera por delante) y su **rendimiento** (lo que hace en el campo).

**Pregunta:** ¿el mercado paga más por la edad o por el rendimiento? ¿Dónde se separan las dos cosas?

**Objetivo:** medir qué pesa más en el valor de mercado y ver si hay jugadores cuyo valor se aleja de lo que sugiere su rendimiento.

## 2. Datos

**Fuente:** [Transfermarkt datasets](https://github.com/dcaribou/transfermarkt-datasets), descargados de Kaggle. Uso tres tablas:

| Tabla | Qué contiene |
|---|---|
| `players` | Datos de cada jugador, entre ellos fecha de nacimiento y posición |
| `player_valuations` | Historial de valores de mercado |
| `appearances` | Cada partido jugado: minutos, goles, asistencias y competición |

**Cobertura:** las valoraciones van del 20 de enero de 2000 al 12 de junio de 2026 y las apariciones, del 3 de julio de 2012 al 28 de junio de 2026. Los datos de Transfermarkt dejan de actualizarse después de junio de 2026, por eso el análisis llega hasta la temporada 2025/26.

**Nota de reproducibilidad:** los CSV no están en el repositorio por su tamaño (`appearances.csv` pesa unos 189 MB). Hay que descargarlos y colocarlos en `src/data/`.

## 3. Exploración inicial

Antes de decidir nada miré qué había en cada tabla.

**Tamaño de cada tabla**

| Tabla | Filas | Jugadores distintos |
|---|---|---|
| `players` | 50.149 | 50.149 |
| `player_valuations` | 656.301 | 41.528 |
| `appearances` | 1.894.350 | 29.531 |

28.328 jugadores aparecen en las tres tablas a la vez, el 96% de los que tienen apariciones. Hay datos de sobra para el análisis.

**Ligas disponibles.** `appearances` mezcla muchas competiciones (al menos 15 con decenas de miles de apariciones). Las cinco grandes encabezan el recuento (`IT1`, `ES1`, `GB1`, `FR1` y `L1`), seguidas por ligas como las de Turquía, Países Bajos o Rusia, que tienen otro nivel y ensuciarían la comparación.

**Posiciones disponibles en `players`**

| Posición | Jugadores |
|---|---|
| Defender | 15.961 |
| Midfield | 14.272 |
| Attack | 13.502 |
| Goalkeeper | 5.828 |
| Missing | 586 |

**Valores nulos relevantes**

- `players`: 49 jugadores sin `date_of_birth`. `position` no tiene nulos. `market_value_in_eur` tiene 8.621 nulos, pero es el valor actual y no se usa en el análisis.
- `player_valuations`: sin nulos en `player_id`, `date` ni valor. `player_club_domestic_competition_id` tiene 92.973 nulos (el 14%), así que el filtro de liga se hace con `competition_id` de `appearances`.
- `appearances`: solo 2 nulos, en `player_name`.

Con esto tomé las decisiones de alcance del paso siguiente.

## 4. Decisiones de alcance

| Decisión | Motivo |
|---|---|
| Cinco ligas: Premier League (`GB1`), LaLiga (`ES1`), Serie A (`IT1`), Bundesliga (`L1`) y Ligue 1 (`FR1`) | Son las grandes ligas europeas, con datos completos y comparables entre sí |
| Solo partidos de liga | Mantiene comparables los minutos, goles y asistencias entre jugadores |
| Solo delanteros y mediocampistas (`Attack` y `Midfield`) | El rendimiento se mide con goles y asistencias, que no describen bien a defensas ni porteros |
| 900 minutos o más por temporada | Para que los goles y asistencias por 90 minutos no dependan de unos pocos partidos |
| Tres temporadas: 2023/24, 2024/25 y 2025/26 | Suficientes observaciones y datos que llegan hasta junio de 2026 |

## 5. Construcción del dataset

Escribí la función `construir_temporada()` (en `src/utils/`), que arma una tabla con una fila por jugador y temporada. La temporada va del 1 de julio al 30 de junio. En resumen:

1. Filtra `appearances` por las cinco ligas y por las fechas de la temporada.
2. Suma por jugador sus minutos, goles y asistencias.
3. Toma la última valoración de mercado de cada jugador dentro de esa temporada.
4. Une rendimiento y valoración (`inner`: solo quedan los jugadores que tienen las dos cosas) y trae de `players` el nombre, la fecha de nacimiento y la posición.
5. Calcula la edad en años al 30 de junio: los días entre la fecha de nacimiento y esa fecha, divididos entre 365,25 para tener en cuenta los años bisiestos.
6. Se queda con delanteros y mediocampistas con 900 minutos o más.
7. Calcula el rendimiento, `ga_90` = (goles + asistencias) / minutos × 90, y añade la columna `temporada`.

Como ejemplo, en 2024/25 salen 2.708 jugadores de las cinco grandes ligas con apariciones, y quedan 825 tras filtrar posición y minutos.

Después uní las tres temporadas con `pd.concat`: **2.579 filas** (873 en 2023/24, 825 en 2024/25 y 881 en 2025/26) y 11 columnas: `player_id`, `minutes_played`, `goals`, `assists`, `market_value_in_eur`, `name`, `date_of_birth`, `position`, `edad`, `ga_90` y `temporada`. La tabla no guarda la liga de cada jugador.

**Variables derivadas**

- `log_valor`: logaritmo en base 10 del valor de mercado. Se usa para que los fichajes muy caros no distorsionen las comparaciones.
- `tramo_edad`, creada con `pd.cut`: menos de 23 (455 filas), 23 a 27 (1.174) y 28 o más (949). Suman 2.578, no 2.579: una fila queda sin tramo de edad porque no tiene fecha de nacimiento.
- `desajuste` (solo en la temporada 2025/26, 881 jugadores): dentro de cada posición, el percentil de valor de mercado menos el percentil de rendimiento (`ga_90`). Va de −1 a 1: cerca de 1, el valor está muy por encima de lo que rinde; cerca de −1, muy por debajo.
- Las 455 filas de menores de 23 son 159 + 145 + 151 por temporada. Un mismo jugador puede aparecer en más de una temporada.

## 6. Hipótesis

| | Hipótesis |
|---|---|
| H1 | En menores de 23, el valor de mercado se asocia más con la edad que con el rendimiento |
| H2 | Pasados los 28, el valor cae aunque el rendimiento se mantenga |
| H3 | El rendimiento explica el valor de forma distinta según la posición, y hay jugadores con un valor muy distinto del que sugiere su rendimiento |

**Alcance de H3.** Los datos no incluyen fama, selección, club ni marketing, que serían las explicaciones de un valor alejado del rendimiento; esas causas solo pueden discutirse como posibles explicaciones. H3 quedó medida en dos partes: la correlación por posición y una lista de casos con valor alejado de su rendimiento (columna `desajuste`).

## 7. Análisis y resultados

### H1 · Menores de 23: refutada

Sobre las 455 observaciones de menores de 23, la correlación con el logaritmo del valor de mercado es:

| Variable | Correlación con el valor |
|---|---|
| Rendimiento (`ga_90`) | 0,49 |
| Edad | −0,20 |

El valor sigue más al rendimiento que a la edad, al revés de lo que suponía la hipótesis.

![H1: valor frente a edad y rendimiento en menores de 23](../presentacion/img/h1_jovenes.png)

### H2 · 28 o más: confirmada

| Tramo de edad | Observaciones | Valor mediano | `ga_90` mediano | Correlación edad y valor | Correlación rendimiento y valor |
|---|---|---|---|---|---|
| Menos de 23 | 455 | 18 M€ | 0,28 | −0,20 | 0,49 |
| 23 a 27 | 1.174 | 15 M€ | 0,32 | −0,11 | 0,45 |
| 28 o más | 949 | 5 M€ | 0,30 | −0,48 | 0,43 |

El valor mediano cae de 15 M€ a 5 M€ a partir de los 28, mientras el rendimiento mediano apenas cambia. Además, en ese tramo la edad (−0,48) pesa tanto como el rendimiento (0,43), cosa que no ocurre en los otros dos tramos.

![H2: valor y rendimiento medianos por tramo de edad](../presentacion/img/h2_veteranos.png)

### H3 · Por posición y casos: confirmada

| Posición | Observaciones | Rendimiento y valor | Edad y valor |
|---|---|---|---|
| Ataque | 1.166 | 0,51 | −0,46 |
| Mediocampo | 1.413 | 0,37 | −0,48 |

En ataque pesa más el rendimiento; en mediocampo pesa más la edad. Una parte de la diferencia es de la medida: goles y asistencias describen mejor a un delantero que a un mediocampista, así que no significa que el mediocampo rinda menos.

![H3: valor frente a rendimiento por posición](../presentacion/img/h3_posicion.png)

**Casos con el valor más alejado de su rendimiento** (temporada 2025/26, columna `desajuste`)

Valor muy por encima de su rendimiento:

| Jugador | Posición | Edad | Valor | `ga_90` | Desajuste |
|---|---|---|---|---|---|
| Carlos Baleba | Midfield | 22,5 | 55 M€ | 0,00 | 0,90 |
| Aurélien Tchouaméni | Midfield | 26,4 | 70 M€ | 0,03 | 0,87 |
| Franco Mastantuono | Attack | 18,9 | 45 M€ | 0,09 | 0,87 |
| Ayyoub Bouaddi | Midfield | 18,7 | 50 M€ | 0,04 | 0,84 |
| Anthony Elanga | Attack | 24,2 | 32 M€ | 0,07 | 0,79 |

Valor muy por debajo de su rendimiento:

| Jugador | Posición | Edad | Valor | `ga_90` | Desajuste |
|---|---|---|---|---|---|
| Michael Gregoritsch | Attack | 32,2 | 1,5 M€ | 0,88 | −0,92 |
| Jonas Hofmann | Midfield | 34,0 | 1,5 M€ | 0,45 | −0,87 |
| Sofiane Boufal | Midfield | 32,8 | 1,8 M€ | 0,44 | −0,84 |
| Andrej Kramaric | Attack | 35,0 | 2,5 M€ | 0,81 | −0,83 |
| Christian Eriksen | Midfield | 34,4 | 2,5 M€ | 0,48 | −0,81 |

En esta lista corta, los más caros que su rendimiento tienen entre 18 y 26 años, y los más baratos, entre 32 y 35. Encaja con H1 y H2: el mercado paga el potencial y descuenta a los veteranos. Son casos individuales, no una prueba, y varios de los más caros son mediocampistas cuyo aporte no se mide con goles y asistencias.

## 8. Conclusiones y limitaciones

**Conclusiones**

1. Entre los menores de 23, el valor sigue más al rendimiento que a la edad (0,49 frente a −0,20).
2. Después de los 28, el valor mediano cae de 15 M€ a 5 M€ con un rendimiento parecido, y la edad pasa a pesar tanto como el rendimiento.
3. En ataque pesa más el rendimiento y en mediocampo la edad, con la salvedad de lo que mide el rendimiento.
4. Los jugadores con más valor del que sugiere su rendimiento son jóvenes y los de menos valor son veteranos.

**Limitaciones**

- **Rendimiento incompleto:** `ga_90` solo cuenta goles y asistencias; en mediocampo se subestima recuperar y distribuir.
- **Solo juegan los que juegan:** con 900 minutos o más, los veteranos que quedan son los que siguen rindiendo, así que su rendimiento sale algo inflado.
- **Observaciones repetidas:** un mismo jugador puede estar en dos o tres temporadas; son 2.579 observaciones, no 2.579 jugadores.
- **Rango de edad estrecho en menores de 23:** la edad varía poco entre los 17 y los 22 años, y parte del −0,20 puede deberse a eso.
- **Asociación, no causa:** una correlación muestra qué se mueve junto, no por qué el mercado paga lo que paga.
- **Casos individuales:** la lista de jugadores con valor alejado de su rendimiento es solo de la temporada 2025/26 e ilustra la diferencia, pero no la demuestra.
- **El valor de estimacion** el valor de Transfermarkt ya incorpora la edad y el potencial del jugador, así que el análisis muestra qué patrones sigue esa estimación, no cómo se comporta el mercado real de traspasos.

**Lo que no se hizo**

- La comparación por liga: la tabla final no guarda la liga de cada jugador.